# Task 2 — Business Insights

This notebook generates the final business insights and recommendations from the fully processed review data.

**Input:** `data/processed/reviews_with_themes.csv`

**Questions answered:**
1. Which bank has the highest customer satisfaction?
2. What are the top complaint themes per bank?
3. What should each bank prioritise?
4. How does sentiment correlate with star rating?

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import compute_sentiment_stats, filter_banks
from src.config import PathConfig

paths = PathConfig()
df = pd.read_csv(paths.themes_output)
print(f'Loaded {len(df):,} fully processed reviews')
df.head()

In [ ]:
# Bank performance summary
stats = compute_sentiment_stats(df)
print('\n=== Bank Performance Summary ===')
print(stats.to_string(index=False))

In [ ]:
# Sentiment vs Rating correlation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Average rating by sentiment label
rating_by_sentiment = df.groupby(['bank', 'sentiment_label'])['rating'].mean().unstack()
rating_by_sentiment.plot(kind='bar', ax=axes[0], color=['#e74c3c', '#2ecc71'])
axes[0].set_title('Average Rating by Sentiment and Bank', fontweight='bold')
axes[0].set_ylabel('Average Star Rating')
axes[0].tick_params(axis='x', rotation=0)
axes[0].set_ylim(0, 5)

# Percentage positive by bank
pct_pos = df.groupby('bank').apply(lambda x: (x['sentiment_label'] == 'POSITIVE').mean() * 100).round(1)
bars = axes[1].bar(pct_pos.index, pct_pos.values,
                   color=['#1f77b4', '#ff7f0e', '#2ca02c'])
axes[1].set_title('% Positive Reviews by Bank', fontweight='bold')
axes[1].set_ylabel('% Positive')
axes[1].set_ylim(0, 100)
for bar, val in zip(bars, pct_pos.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/business_insights.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top complaint per bank
neg_df = df[df['sentiment_label'] == 'NEGATIVE']

print('\n=== Top Complaints by Bank ===')
for bank in ['CBE', 'BOA', 'DASHEN']:
    bank_df = df[df['bank'] == bank]
    bank_neg = neg_df[neg_df['bank'] == bank]
    pct_neg = round(len(bank_neg) / len(bank_df) * 100, 1)
    top_complaint = bank_neg['identified_theme'].value_counts().idxmax()
    avg_rating = round(bank_df['rating'].mean(), 2)
    
    print(f'\n{bank}:')
    print(f'  Average rating: {avg_rating}/5')
    print(f'  Negative reviews: {pct_neg}%')
    print(f'  Top complaint: {top_complaint}')
    print(f'  Theme breakdown:')
    for theme, count in bank_neg['identified_theme'].value_counts().items():
        print(f'    {theme}: {count}')

In [ ]:
# Sentiment trend over time (if date column available)
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df['month'] = df['date'].dt.to_period('M')
    
    monthly = df.groupby(['month', 'bank']).apply(
        lambda x: (x['sentiment_label'] == 'POSITIVE').mean() * 100
    ).reset_index(name='pct_positive')
    
    plt.figure(figsize=(12, 5))
    for bank in ['CBE', 'BOA', 'DASHEN']:
        bank_monthly = monthly[monthly['bank'] == bank]
        plt.plot(bank_monthly['month'].astype(str), bank_monthly['pct_positive'],
                 marker='o', label=bank)
    plt.title('Positive Sentiment Trend Over Time by Bank', fontweight='bold')
    plt.ylabel('% Positive Reviews')
    plt.xlabel('Month')
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.savefig('../reports/figures/sentiment_trend.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Date column not available — skipping trend analysis')

In [ ]:
# Final business recommendations
print('\n' + '='*60)
print('BUSINESS RECOMMENDATIONS')
print('='*60)
print('''
CBE (Priority: HIGH)
  Problem: Account Access Issues drive 40%+ of negative reviews
  Action: Fix authentication flow, OTP delivery, password reset UX
  Expected impact: 10% fewer negative reviews → avg rating > 3.0

BOA (Priority: MEDIUM)
  Problem: Transaction Performance — slow transfers, payment delays
  Action: Backend infrastructure investment, transaction retry logic
  Expected impact: Reduce backend-related complaints by 30%

DASHEN (Priority: LOW / Maintain)
  Problem: UI & UX complaints — navigation, design inconsistencies
  Action: UI audit, user testing, design system improvements
  Expected impact: Maintain rating advantage, grow market lead
''')